# Stage 7 · Finish dense Lego and score it in one session

Attach the original Stage 0 kit, the reviewed **lego-dense-vanilla-s42-stage6-review.zip** and the offline **stage4-lpips.zip**. Select **GPU T4 ×2** and run all cells. On one T4, the measured estimate is 40 minutes of training and 8 minutes of evaluation, plus setup and exports.

The training cell resumes at 60,000 and saves portable snapshots at 65,000, 70,000 and 75,000. The scoring cell uses the completed checkpoint and the frozen 25 held-out cameras. A Stage 7 checkpoint ZIP or expanded snapshot can replace the Stage 6 input in a fresh session. A score or download failure can therefore be retried without repeating the fit. Send **lego-dense-vanilla-s42-stage7-review.zip**, including if the dense result falls outside the reference window.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import base64
import hashlib
import json
import math
import os
import shutil
import statistics
import subprocess
import sys
import time
import zipfile

import torch
from IPython.display import HTML, display
from PIL import Image

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
root = outputs / "recovery-stage7"
root.mkdir(parents=True, exist_ok=True)
if "stage7_temp" not in globals():
    stage7_temp = TemporaryDirectory(prefix="dense-stage7-")
cache = Path(stage7_temp.name)
relative_run = Path("runs/lego/dense-vanilla/seed-42")
run_id = "0aee2c9d8be2e3010c28850a68f3d61a"
start_sha = "8d60767ce8eded78af78421ee0a0daecf789a71763d11eb0c84c8a071cfd1ba1"
meta_sha = "3c26db45b01569fb32aa76daff5bcfcf92920490c5a86173d791ed74699a6842"
data_hash = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
expected_ids = [f"test/r_{i}.png" for i in range(0, 200, 8)]
start_step, endpoint = 60_000, 75_000
device = torch.device("cuda:0")
if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU and restart this notebook.")
hardware = dict(torch=str(torch.__version__), cuda=torch.version.cuda, device=str(device),
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def unpack(path):
    folder = cache / hashlib.sha256(str(path).encode()).hexdigest()
    folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not (folder / name).resolve().is_relative_to(folder.resolve()):
                raise ValueError(f"Unexpected member {name}")
            archive.extract(name, folder)
    return folder

def kit(marker, filename):
    roots = sorted(inputs.rglob(marker))
    if roots:
        return roots[0].parent
    archives = sorted(inputs.rglob(filename))
    if not archives:
        raise FileNotFoundError(f"Attach {filename} or its expanded dataset.")
    return unpack(archives[0])

repo = kit("RECOVERY_STAGE0.txt", "stage0-input.zip")
metric_kit = kit("RECOVERY_LPIPS.txt", "stage4-lpips.zip")
print("GPU", hardware["devices"][0])

In [ ]:
sys.path.insert(0, str(repo))
from recovery.data import verify_manifest
from recovery.media import digest, model_fingerprint
from recovery.training import implementation_hash, load_state, make_model, settings, train, write_json

assert implementation_hash() == code_hash
score_sources = {
    "recovery/media.py": "fd13945accd1ec6938b7ebceccec31fe1f8716dc2a2ad51d84068d560e09a3af",
    "sparse_nerf/evaluate.py": "eb2a0c14ab7aac4de41fc5de9e2e2316ba902e08a14cc2b1a06d7616171e58ca",
}
for name, expected in score_sources.items():
    assert hashlib.sha256((repo / name).read_text(encoding="utf-8").encode()).hexdigest() == expected

def check_run(folder, expected_step=None):
    path = folder / relative_run
    if sha(path / "run.json") != meta_sha:
        raise ValueError("The run metadata differs from the reviewed continuation.")
    meta = read_json(path / "run.json")
    state = load_state(path, meta)
    step = state["step"]
    assert start_step <= step <= endpoint and (expected_step is None or step == expected_step)
    slots = state["optimizer"]["state"]
    assert len(slots) == 48 and all(s["step"].item() == step for s in slots.values())
    assert all(g["lr"] == 0.00005 for g in state["optimizer"]["param_groups"])
    tensors = [*state["model"].values(), *(s[k] for s in slots.values() for k in ("exp_avg", "exp_avg_sq"))]
    assert all(t.isfinite().all().item() for t in tensors)
    assert all(isinstance(state[k], torch.Tensor) and state[k].numel() for k in ("cpu_rng", "cuda_rng"))
    return step

def candidate(record, marker, path):
    step, checkpoint_sha = record.get("checkpoint_step"), record.get("checkpoint_sha256")
    if record.get("run_id") != run_id or not isinstance(step, int):
        return None
    if marker == "stage6-review.json":
        if (step, checkpoint_sha) != (start_step, start_sha):
            return None
    elif record.get("stage6_checkpoint_sha256") != start_sha or not start_step <= step <= endpoint:
        return None
    return step, int(record.get("scored") is True), 0, path, checkpoint_sha

choices = []
for marker, pattern in (("stage6-review.json", "lego-dense-vanilla-s42-stage6-review.zip"),
                        ("stage7-review.json", "lego-dense-vanilla-s42-stage7-*.zip")):
    for path in inputs.rglob(marker):
        choices.append(candidate(read_json(path), marker, path.parent))
    for path in inputs.rglob(pattern):
        with zipfile.ZipFile(path) as archive:
            choices.append(candidate(json.loads(archive.read(marker)), marker, path))
choices = [c for c in choices if c is not None]
for folder in root.glob("from-*"):
    ckpt = folder / relative_run / "checkpoint.pt"
    if ckpt.is_file():
        step = check_run(folder)
        choices.append((step, int((ckpt.parent / "eval/metrics.json").is_file()), 1, folder, sha(ckpt)))
if not choices:
    raise FileNotFoundError("Attach the Stage 6 review or a Stage 7 snapshot.")
step, scored, local, source, checkpoint_sha = max(choices)
if len({c[4] for c in choices if c[0] == step}) != 1:
    raise ValueError("Different checkpoints claim the same update. Attach the intended continuation.")
if not local and source.is_file():
    source = unpack(source)
if not local:
    marker = "stage7-review.json" if (source / "stage7-review.json").is_file() else "stage6-review.json"
    hashes = read_json(source / marker)["artifact_sha256"]
    required = {"manifests/lego.json", (relative_run / "run.json").as_posix(), (relative_run / "checkpoint.pt").as_posix()}
    if scored:
        required.add((relative_run / "eval/metrics.json").as_posix())
        for index in (0, 12, 24):
            required.update((relative_run / "eval" / f"view-{index:03d}-{kind}.png").as_posix()
                            for kind in ("rgb", "target", "depth", "opacity"))
    assert required <= hashes.keys()
    for name, expected_sha in hashes.items():
        member = (source / name).resolve()
        assert member.is_relative_to(source.resolve()) and sha(member) == expected_sha
assert sha(source / relative_run / "checkpoint.pt") == checkpoint_sha
check_run(source, step)
frozen = verify_manifest(source / "manifests/lego.json", repo / "data/blender/lego")
assert frozen["data_hash"] == data_hash
assert [frozen["frames"][i]["file_path"] for i in frozen["splits"]["test"]] == expected_ids
work = source if local else root / f"from-{step}-{checkpoint_sha[:12]}{'-scored' if scored else ''}"
run = work / relative_run
manifest = work / "manifests/lego.json"
if not local and not work.exists():
    with TemporaryDirectory(dir=root) as temporary:
        fresh = Path(temporary) / "run-root"
        fresh_run = fresh / relative_run
        fresh_run.mkdir(parents=True)
        # Keep the input snapshots read-only. A scored Stage 7 export includes its reusable eval directory.
        for p in (source / relative_run).iterdir():
            if p.is_file() and (p.name in {"run.json", "checkpoint.pt", "health.json", "training.jsonl"} or
                                p.match("train-*.png")):
                shutil.copyfile(p, fresh_run / p.name)
        if scored:
            shutil.copytree(source / relative_run / "eval", fresh_run / "eval")
            if (source / "evaluation.log").is_file():
                shutil.copyfile(source / "evaluation.log", fresh / "evaluation.log")
        (fresh / "manifests").mkdir()
        write_json(fresh / "manifests/lego.json", frozen)
        check_run(fresh, step)
        fresh.rename(work)
if read_json(manifest) != frozen:
    write_json(manifest, frozen)
assert sha(run / "checkpoint.pt") == checkpoint_sha
meta = read_json(run / "run.json")
prefix_cfg = settings("main", frozen["camera"])
cfg = dict(prefix_cfg, steps=endpoint, lr=0.00005, lr_final=0.00005,
           prefix_config=prefix_cfg, prefix_updates=50000, schedule="50k-exponential-then-25k-hold-v1")
assert meta["training_config"] == cfg and meta["run_id"] == run_id and meta["data_hash"] == data_hash
print(f"Restored step {step:,}. Training ends at {endpoint:,}, then evaluation scores {len(expected_ids)} views.")

In [ ]:
# Install the pinned LPIPS wheel in a stage-local directory. Kaggle supplies torch and torchvision.
wheel = metric_kit / "wheels/lpips-0.1.4-py3-none-any.whl"
torch_cache = metric_kit / "torch-cache"
weights = torch_cache / "hub/checkpoints/alexnet-owt-7be5be79.pth"
wheel_hash = sha(wheel)
weight_hash = sha(weights)
assert wheel_hash == "fd537af5828b69d2e6ffc0a397bd506dbc28ca183543617690844c08e102ec5e"
assert weight_hash == "7be5be791159472b1fbf3c69796f7cb30dca7ad8466c2df70058c37116cdee02"
site = work / "metric-packages"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--upgrade",
                "--target", str(site), str(wheel)], check=True)
env = os.environ.copy()
env.update(TORCH_HOME=str(torch_cache), PYTHONPATH=os.pathsep.join([str(site), str(repo), env.get("PYTHONPATH", "")]))
probe = """
import importlib.metadata, json, torch, torchvision
from sparse_nerf.evaluate import lpips_model
from recovery.media import model_fingerprint
net = lpips_model().eval()
with torch.inference_mode():
    sample = torch.zeros(1, 3, 64, 64)
    identical = net(sample, sample).item()
    different = net(sample, sample + 1).item()
assert identical == 0.0 and different > 0 and torch.isfinite(torch.tensor(different))
print(json.dumps(dict(lpips=importlib.metadata.version('lpips'), torchvision=torchvision.__version__,
                      lpips_model_sha256=model_fingerprint(net), identical=identical, different=different)))
"""
probe_output = subprocess.check_output([sys.executable, "-c", probe], cwd=repo, env=env, text=True)
lpips_info = json.loads(probe_output.strip().splitlines()[-1])
assert lpips_info["lpips"] == "0.1.4"
assert lpips_info["lpips_model_sha256"] == "913e6288d265ca661f26fe0426b840bc69bafcf9fcda0fb2c99586891562ea9e"
print("Offline LPIPS-Alex v0.1 ready.", lpips_info)

In [ ]:
def export(step, scored=False):
    check_run(work, step)
    files = [manifest, run / "run.json", run / "checkpoint.pt"]
    health = run / "health.json"
    if health.exists() and read_json(health)["step"] == step:
        files.append(health)
    files.extend(p for p in run.glob("train-*.png") if int(p.stem.split("-")[1]) <= step)
    receipt = dict(run_id=run_id, checkpoint_step=step, stage6_checkpoint_sha256=start_sha,
                   checkpoint_sha256=sha(run / "checkpoint.pt"), data_hash=data_hash,
                   training_code_sha256=code_hash, hardware=hardware, scored=scored)
    if scored:
        assert step == endpoint
        files.extend(checked_eval_files)
        if (work / "evaluation.log").is_file():
            files.append(work / "evaluation.log")
        receipt.update(mean=result["mean"], test_views=len(result["frames"]), test_seconds=result["test_seconds"],
                       comparison_hash=result["comparison_hash"], model_sha256=result["model_sha256"],
                       dense_lego_reference=result["dense_lego_reference"], lpips=lpips_info,
                       eval_code_sha256=score_sources, lpips_wheel_sha256=wheel_hash, alexnet_weights_sha256=weight_hash)
    tag = "review" if scored else f"step{step}"
    target = outputs / f"lego-dense-vanilla-s42-stage7-{tag}.zip"
    pending = target.with_suffix(".tmp")
    hashes = {p.relative_to(work).as_posix(): sha(p) for p in files}
    with zipfile.ZipFile(pending, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for p in files:
            archive.write(p, p.relative_to(work).as_posix())
        log = run / "training.jsonl"
        if log.is_file():
            committed = []
            for line in log.read_text(encoding="utf-8").splitlines():
                try:
                    logged_step = json.loads(line)["step"]
                except (ValueError, KeyError):
                    continue
                if logged_step <= step:
                    committed.append(line)
            blob = ("\n".join(committed) + "\n").encode()
            name = (relative_run / log.name).as_posix()
            hashes[name] = hashlib.sha256(blob).hexdigest()
            archive.writestr(name, blob)
        receipt["artifact_sha256"] = hashes
        archive.writestr("stage7-review.json", json.dumps(receipt, allow_nan=False))
    with zipfile.ZipFile(pending) as archive:
        assert set(archive.namelist()) == set(hashes) | {"stage7-review.json"} and archive.testzip() is None
    os.replace(pending, target)
    print(f"Saved {target.name}  SHA-256 {sha(target)}", flush=True)
    return target

def download(path):
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{path.name}" href="data:application/zip;base64,{encoded}">Download {path.name}</a>'))

def finish_training():
    current = check_run(work)
    for stop in (65_000, 70_000, endpoint):
        if current < stop:
            try:
                train(run, manifest, "dense-vanilla", 42, device, cfg, resume=True, stop_at=stop, save_every=500)
            except BaseException:
                download(export(check_run(work)))
                raise
            current = check_run(work, stop)
            saved = export(current)
            # The Output pane keeps every milestone. The embedded copy protects the final checkpoint link.
            if current == endpoint:
                download(saved)
    print(f"Ready to score the completed {current:,}-update checkpoint.")

finish_training()

In [ ]:
metrics_path = run / "eval/metrics.json"
chunk = 1024
def score():
    command = [sys.executable, "-u", "-m", "recovery.media", "eval", "--run", str(run),
               "--manifest", str(manifest), "--device", "cuda:0", "--chunk", str(chunk)]
    log_path = work / "evaluation.log"
    first_view_time = None
    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
        command, cwd=repo, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                if line.startswith("[test "):
                    number = int(line.split("/", 1)[0].split()[-1])
                    if number == 1:
                        first_view_time = time.monotonic()
                    elif first_view_time is not None:
                        seconds = (time.monotonic() - first_view_time) / (number - 1)
                        print(f"Measured {seconds:.1f} s/view; about {(25-number)*seconds/60:.1f} min left.", flush=True)
            if process.wait():
                raise RuntimeError(f"Evaluation exited with status {process.returncode}. See {log_path}.")
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
if metrics_path.exists():
    print("Found completed metrics in this session. Checking them against the inputs below.")
else:
    score()

In [ ]:
result = read_json(metrics_path)
compare = result["comparison"]
rows = result["frames"]
state = load_state(run, meta)
assert state["step"] == endpoint
model = make_model(cfg, torch.device("cpu"))
model.load_state_dict(state["model"])
assert result["model_sha256"] == model_fingerprint(model)
del state, model
assert (result["run_id"], result["scene"], result["protocol"], result["seed"], result["train_views"]) == (
    run_id, "lego", "dense-vanilla", 42, 100)
assert compare["data_hash"] == data_hash and compare["training_config"] == cfg
assert compare["lpips_weights_sha256"] == lpips_info["lpips_model_sha256"]
assert compare["test_ids"] == [row["frame"] for row in rows] == expected_ids
assert (compare["width"], compare["height"], compare["white_background"]) == (400, 400, True)
assert compare["target_hash"] == "d749094281b2181931225c785bd6914831a0fae89394dd212d7c0ade5e1bb26f"
assert result["comparison_hash"] == digest(compare)
assert math.isfinite(result["test_seconds"]) and result["test_seconds"] > 0
for key in ("psnr", "ssim", "lpips", "white_psnr", "foreground_psnr", "white_foreground_psnr", "foreground_opacity"):
    values = [row[key] for row in rows]
    assert all(map(math.isfinite, values)) and math.isclose(statistics.mean(values), result["mean"][key], abs_tol=1e-8)
assert math.isclose(result["mean"]["white_psnr"], 9.492596382193124, abs_tol=1e-5)
reference = result["dense_lego_reference"]
assert reference["range_db"] == [30.54, 34.54] and reference["inside_range"] == (30.54 <= result["mean"]["psnr"] <= 34.54)
checked_eval_files = [metrics_path]
for index in (0, 12, 24):
    for kind in ("rgb", "target", "depth", "opacity"):
        name = f"view-{index:03d}-{kind}.png"
        assert rows[index]["artifacts"][kind] == name
        path = metrics_path.parent / name
        with Image.open(path) as picture:
            assert picture.format == "PNG" and picture.size == (400, 400)
            picture.verify()
        checked_eval_files.append(path)
print(f"Held-out PSNR {result['mean']['psnr']:.3f} dB, SSIM {result['mean']['ssim']:.4f}, LPIPS-Alex {result['mean']['lpips']:.5f}.")
print(f"C4 range 30.54–34.54 dB. Inside range: {reference['inside_range']}.")
download(export(endpoint, scored=True))